In [26]:
import fastf1
import pandas as pd
from rapidfuzz.process_cpp_impl import INT64

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

fastf1.Cache.enable_cache("ff1_cache")

In [27]:
rounds = [20, 21, 22, 23, 24]

all_laps = []
all_results = []
all_events = []

for r in rounds:
    session = fastf1.get_session(2025, r, "R")
    session.load(laps=True, telemetry=False, weather=False, messages=False)
    race_laps = session.laps
    race_laps["round_number"] = r
    all_laps.append(race_laps)
    all_results.append(session.results)
    all_events.append(session.event)

laps = pd.concat(all_laps, ignore_index=True)
results = pd.concat(all_results, ignore_index=True)
events = pd.concat(all_events, axis=1).T

print(len(laps))
print(len(results))
print(laps["TrackStatus"].value_counts())

my_rule = (
    (laps["LapNumber"] == 1)
    | laps["PitInTime"].notna()
    | laps["PitOutTime"].notna()
    | (laps["TrackStatus"] != "1")
)
print("mine:", my_rule.sum())
print("fastf1 flags but I miss:", (~laps["IsAccurate"] & ~my_rule).sum())
print("I flag but fastf1 accepts:", (laps["IsAccurate"] & my_rule).sum())

missed = laps[~laps["IsAccurate"] & ~my_rule]
print(len(missed))
print(missed["TrackStatus"].value_counts())
print(missed["LapTime"].isna().sum())
print(missed["Deleted"].value_counts())

extra = laps[laps["IsAccurate"] & my_rule & (laps["TrackStatus"] != "1") & laps["PitInTime"].isna() & laps["PitOutTime"].isna() & (laps["LapNumber"] != 1)]
print(len(extra))
print(extra["TrackStatus"].value_counts())

print(laps["round_number"].value_counts())

core           INFO 	Loading data for Mexico City Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['4', '16', '1', '87', '81', '12', '63', '44', '31', '5', '22', '23', '6', '18', '10', '43', '55', '14', '27', '30']
core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data 

5623
100
TrackStatus
1      5220
12      106
4        76
671      49
41       38
26       31
126      23
124      20
24       19
167      11
67       10
71       10
16        7
6         2
216       1
Name: count, dtype: int64
mine: 721
fastf1 flags but I miss: 0
I flag but fastf1 accepts: 64
0
Series([], Name: count, dtype: int64)
0
Series([], Name: count, dtype: int64)
64
TrackStatus
12    64
Name: count, dtype: int64
round_number
20    1263
21    1251
24    1156
23    1067
22     886
Name: count, dtype: int64


In [28]:
print(results["Abbreviation"].nunique())
print(laps["Driver"].nunique())

20
20


In [29]:
dim_team = (
    laps[["Team"]].drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={
        "Team": "team_name",
    })
)
dim_team.insert(0, "team_key", range(1, len(dim_team) + 1))
dim_team

,team_key,team_name
0,1,Red Bull Racing
1,2,Alpine
2,3,Mercedes
3,4,Aston Martin
4,5,Ferrari
5,6,Williams
6,7,Kick Sauber
7,8,Racing Bulls
8,9,Haas F1 Team
9,10,McLaren


In [30]:
print(laps.columns.tolist())

['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate', 'round_number']


In [31]:
dim_driver = (
    results[["Abbreviation", "FullName", "DriverNumber"]].drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={
        "Abbreviation": "driver_code",
        "FullName": "driver_name",
        "DriverNumber": "driver_number"
    })
)
dim_driver.insert(0, "driver_key", range(1, len(dim_driver) + 1))
dim_driver

,driver_key,driver_code,driver_name,driver_number
0,1,NOR,Lando Norris,4
1,2,LEC,Charles Leclerc,16
2,3,VER,Max Verstappen,1
3,4,BEA,Oliver Bearman,87
4,5,PIA,Oscar Piastri,81
5,6,ANT,Kimi Antonelli,12
6,7,RUS,George Russell,63
7,8,HAM,Lewis Hamilton,44
8,9,OCO,Esteban Ocon,31
9,10,BOR,Gabriel Bortoleto,5


In [32]:
print(pd.concat(all_events, axis=1).T)
events = pd.concat(all_events, axis=1).T

   RoundNumber               Country     Location                                  OfficialEventName            EventDate               EventName        EventFormat    Session1  \
20          20                Mexico  Mexico City  FORMULA 1 GRAN PREMIO DE LA CIUDAD DE MÉXICO 2025  2025-10-26 00:00:00  Mexico City Grand Prix       conventional  Practice 1   
21          21                Brazil    São Paulo  FORMULA 1 MSC CRUISES GRANDE PRÊMIO DE SÃO PAU...  2025-11-09 00:00:00    São Paulo Grand Prix  sprint_qualifying  Practice 1   
22          22         United States    Las Vegas       FORMULA 1 HEINEKEN LAS VEGAS GRAND PRIX 2025  2025-11-22 00:00:00    Las Vegas Grand Prix       conventional  Practice 1   
23          23                 Qatar       Lusail      FORMULA 1 QATAR AIRWAYS QATAR GRAND PRIX 2025  2025-11-30 00:00:00        Qatar Grand Prix  sprint_qualifying  Practice 1   
24          24  United Arab Emirates   Yas Island  FORMULA 1 ETIHAD AIRWAYS ABU DHABI GRAND PRIX ...

In [33]:
dim_session = (
    events[["RoundNumber", "Country", "Location", "EventName", "EventFormat", "EventDate"]]
    .reset_index(drop=True)
    .rename(columns={
        "RoundNumber": "round_number",
        "Country": "country",
        "Location": "location",
        "EventName": "event_name",
        "EventFormat": "event_format",
        "EventDate": "event_date"
    })
)
dim_session["year"] = 2025
dim_session["session_type"] = "R"
dim_session.insert(0, "session_key", range(1, len(dim_session) + 1))
dim_session

,session_key,round_number,country,location,event_name,event_format,event_date,year,session_type
0,1,20,Mexico,Mexico City,Mexico City Grand Prix,conventional,2025-10-26 00:00:00,2025,R
1,2,21,Brazil,São Paulo,São Paulo Grand Prix,sprint_qualifying,2025-11-09 00:00:00,2025,R
2,3,22,United States,Las Vegas,Las Vegas Grand Prix,conventional,2025-11-22 00:00:00,2025,R
3,4,23,Qatar,Lusail,Qatar Grand Prix,sprint_qualifying,2025-11-30 00:00:00,2025,R
4,5,24,United Arab Emirates,Yas Island,Abu Dhabi Grand Prix,conventional,2025-12-07 00:00:00,2025,R


In [34]:
print(laps["Compound"].unique())

['MEDIUM' 'SOFT' 'HARD']


In [35]:
dim_compound = pd.DataFrame({
    "compound_key": [1, 2, 3, 4, 5],
    "compound_name": ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"],
})

In [40]:
fct = laps.merge(dim_team, left_on="Team", right_on="team_name", how="left")
fct = fct.merge(dim_driver, left_on="Driver", right_on="driver_code", how="left")
fct = fct.merge(dim_compound, left_on="Compound", right_on="compound_name", how="left")
fct = fct.merge(dim_session[["session_key","round_number"]], on="round_number", how="left")

fct = fct.drop(columns=["Driver", "DriverNumber", "Team", "Compound", "team_name", "driver_code", "driver_name", "driver_number", "compound_name", "FastF1Generated", "round_number"])

fct["is_first_lap"] = fct["LapNumber"] == 1
fct["is_in_lap"] = fct["PitInTime"].notna()
fct["is_out_lap"] = fct["PitOutTime"].notna()
fct["is_not_green"] = fct["TrackStatus"] != "1"
fct["is_pace_lap"] = ~(fct["is_first_lap"] | fct["is_in_lap"] | fct["is_out_lap"] | fct["is_not_green"])

fct = fct.astype({
    "LapNumber": "Int64",
    "Stint": "Int64",
    "TyreLife": "Int64",
    "Position": "Int64"
})

fct = fct.rename(columns={"Time": "time",
                    "LapTime": "lap_time",
                    "LapNumber": "lap_number",
                    "Stint": "stint",
                    "PitOutTime": "pit_out_time",
                    "PitInTime": "pit_in_time",
                    "Sector1Time": "sector1_time",
                    "Sector2Time": "sector2_time",
                    "Sector3Time": "sector3_time",
                    "SpeedI1": "speed_i1",
                    "SpeedI2": "speed_i2",
                    "SpeedFL": "speed_fl",
                    "SpeedST": "speed_st",
                    "Sector1SessionTime": "sector1_session_time",
                    "Sector2SessionTime": "sector2_session_time",
                    "Sector3SessionTime": "sector3_session_time",
                    "IsPersonalBest": "is_personal_best",
                    "TyreLife": "tyre_life",
                    "FreshTyre": "fresh_tyre",
                    "LapStartTime": "lap_start_time",
                    "LapStartDate": "lap_start_date",
                    "TrackStatus": "track_status",
                    "Position": "position",
                    "Deleted": "deleted",
                    "DeletedReason": "deleted_reason",
                    "IsAccurate": "is_accurate",

                    })

print(fct.columns.tolist())
print(fct["is_pace_lap"].sum())
print(len(fct))
print(fct[["lap_number", "stint", "tyre_life", "position"]].dtypes)
print(fct[["team_key", "driver_key", "compound_key"]].isna().sum())

['time', 'lap_time', 'lap_number', 'stint', 'pit_out_time', 'pit_in_time', 'sector1_time', 'sector2_time', 'sector3_time', 'sector1_session_time', 'sector2_session_time', 'sector3_session_time', 'speed_i1', 'speed_i2', 'speed_fl', 'speed_st', 'is_personal_best', 'tyre_life', 'fresh_tyre', 'lap_start_time', 'lap_start_date', 'track_status', 'position', 'deleted', 'deleted_reason', 'is_accurate', 'team_key', 'driver_key', 'compound_key', 'session_key', 'is_first_lap', 'is_in_lap', 'is_out_lap', 'is_not_green', 'is_pace_lap']
4902
5623
lap_number    Int64
stint         Int64
tyre_life     Int64
position      Int64
dtype: object
team_key        0
driver_key      0
compound_key    0
dtype: int64


In [39]:
print(len(fct))
print(fct[["team_key", "driver_key", "compound_key", "session_key"]].isna().sum())

5623
team_key        0
driver_key      0
compound_key    0
session_key     0
dtype: int64
